# Paired 10x Multiome integration (scBIOT 1.2.0)

The published object contains paired RNA counts and ATAC gene activity for the same cells. The gene-activity feature names are stored with the matrix, eliminating the old notebook's undeclared cache files.

- Data: [GSE194122 BMMC Multiome AnnData](https://ndownloader.figshare.com/files/59742665)
- Original study: [GEO GSE194122](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE194122)

The default uses 10,000 paired cells. Set `SCBIOT_TUTORIAL_MAX_CELLS=0` for the full object.


In [ ]:
from pathlib import Path
import os
import urllib.request
import numpy as np
import scanpy as sc
import scbiot as scb

RANDOM_STATE = 0
ROOT = Path(os.environ.get("SCBIOT_TUTORIALS_PATH", Path.cwd())).resolve()
if ROOT.name == "R":
    ROOT = ROOT.parent
DATA_DIR = Path(os.environ.get("SCBIOT_TUTORIAL_DATA", ROOT / "inputs")).resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)

def fetch(filename, url):
    path = DATA_DIR / filename
    if path.exists():
        return path
    partial = path.with_suffix(path.suffix + ".part")
    print(f"Downloading {url} -> {path}")
    urllib.request.urlretrieve(url, partial)
    partial.replace(path)
    return path

def subsample(adata, default_max):
    # Raw is not needed here and can prevent indexed reads from backed sparse files.
    if getattr(adata, "isbacked", False) and adata.raw is not None:
        adata.raw = None
    max_cells = int(os.environ.get("SCBIOT_TUTORIAL_MAX_CELLS", default_max))
    if max_cells > 0 and adata.n_obs > max_cells:
        rng = np.random.default_rng(RANDOM_STATE)
        keep = np.sort(rng.choice(adata.n_obs, max_cells, replace=False))
        return adata[keep].to_memory() if getattr(adata, "isbacked", False) else adata[keep].copy()
    return adata.to_memory() if getattr(adata, "isbacked", False) else adata.copy()

AE_EPOCHS = int(os.environ.get("SCBIOT_AE_EPOCHS", "30"))
USE_GPU = os.environ.get("SCBIOT_USE_GPU", "0") == "1"


In [ ]:
import anndata as ad
import pandas as pd
from scipy import sparse
path = fetch("multiome.h5ad", "https://ndownloader.figshare.com/files/59742665")
source = sc.read_h5ad(path)
paired = subsample(source, 10_000)
gex_mask = paired.var["feature_types"].astype(str).eq("GEX").to_numpy()
adata_gex = paired[:, gex_mask].copy()
adata_gex.X = adata_gex.layers["counts"].copy()
adata_gex.layers["counts"] = adata_gex.X.copy()
ga_names = np.asarray(paired.uns["ATAC_gene_activity_var_names"], dtype=str)
ga_matrix = paired.obsm["ATAC_gene_activity"]
ga_matrix = (
    ga_matrix.tocsr().astype(np.float32)
    if sparse.issparse(ga_matrix)
    else sparse.csr_matrix(np.asarray(ga_matrix, dtype=np.float32))
)
adata_ga = ad.AnnData(
    X=ga_matrix,
    obs=paired.obs.copy(), var=pd.DataFrame(index=ga_names),
)
adata_ga.layers["ga_smooth"] = adata_ga.X.copy()
adata_gex.shape, adata_ga.shape


## Co-embed, integrate, and transfer RNA labels to ATAC


In [ ]:
adata_gex.obs_names = "rna:" + adata_gex.obs_names.astype(str)
adata_ga.obs_names = "atac:" + adata_ga.obs_names.astype(str)
adata = scb.pp.autoencoder_map(
    adata_gex, adata_ga, label="modality", keys=("reference", "query"),
    reference_layer="counts", query_layer="ga_smooth",
    label_key="cell_type", unlabeled_category="Unknown", out_key="X_ae",
    n_top_genes=3000, latent_dim=30, max_epochs=AE_EPOCHS,
    early_stop_patience=5, random_state=RANDOM_STATE,
)
adata, metrics = scb.ot.integrate(
    adata, obsm_key="X_ae", batch_key="modality", out_key="X_supbiot",
    label_key="cell_type", unlabeled_category="Unknown",
    prealign="ot", prealign_strength=0.8, align_reference=True,
    random_state=RANDOM_STATE,
    use_gpu=USE_GPU,
)
adata = scb.ot.supbiot(
    adata, use_rep="X_supbiot", input_rep_key="X_ae",
    label_key="cell_type", unlabeled_category="Unknown", min_conf=0.0,
    random_state=RANDOM_STATE,
    use_gpu=USE_GPU,
)
metrics


In [ ]:
sc.pp.neighbors(adata, use_rep="X_supbiot", random_state=RANDOM_STATE)
sc.tl.umap(adata, random_state=RANDOM_STATE)
sc.pl.umap(adata, color=["modality", "cell_type", "pred_cell_type"])
